# FeSEM — standalone Kaggle baseline

All model definitions, local updates, server operations and experiment execution are included below.
Enable **Internet** and a **T4 GPU**, then choose **Run All**.

The experiment grid covers **CIFAR-10 and EMNIST Balanced**, Dirichlet concentrations
**0.3, 0.5 and 0.7**, and seeds **42, 101, 2024, 3407 and 8888**.
Every communication round trains **all 20 clients**, with **100 rounds**, **one local epoch**
and **batch size 32**. The data partitions, validation holdout, CNN initialization and
minibatch streams are shared with the LatentFed notebooks.

Use the experiment-selection cell to choose a dataset, concentration and seed batch.
Results and checkpoints are stored locally and exported as a ZIP. No companion source
files, repository clone, account token or cloud storage connection is required.

## 1. Dependencies

Install numerical dependencies before importing the method.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import importlib.metadata

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
for package in ("torch", "torchvision"):
    if importlib.util.find_spec(package) is None:
        raise RuntimeError("Select a Kaggle GPU runtime with torch and torchvision, then restart.")
dependencies = [
    ("numpy", "numpy", "2.1.3"),
    ("pandas", "pandas", "2.3.3"),
    ("scipy", "scipy", "1.16.3"),
    ("sklearn", "scikit-learn", "1.6.1"),
    ("hdbscan", "hdbscan", "0.8.44"),
]
install, restart_packages = [], []
for module, distribution, version in dependencies:
    try:
        current_version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        install.append(f"{distribution}=={version}")
        if module in sys.modules:
            restart_packages.append(module)
for module, requirement in [("matplotlib", "matplotlib>=3.9,<3.11"), ("tqdm", "tqdm>=4.66,<5")]:
    if importlib.util.find_spec(module) is None:
        install.append(requirement)
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])
if restart_packages:
    raise RuntimeError(
        "Dependencies were updated. Restart the kernel and Run All: " + ", ".join(restart_packages)
    )
print("Dependencies ready. All method code is contained in the cells below.")


## 2. Imports

Standard Python and installed scientific packages provide the runtime.

In [ ]:
from pathlib import Path
import copy
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import random
import time
import types
import zipfile
from dataclasses import asdict, dataclass, replace
from contextlib import contextmanager
from typing import Any
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
)
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from tqdm.auto import tqdm
from IPython.display import display, FileLink
import hdbscan
import matplotlib.pyplot as plt

PAPER_DATASETS = ("cifar10", "emnist")
PAPER_ALPHAS = (0.3, 0.5, 0.7)
PAPER_SEEDS = (42, 101, 2024, 3407, 8888)
SUPPORTED_VARIANTS = ("full", "no_mdfe", "no_cald", "ae_only")
DEV_SEEDS = [7, 19]
FINAL_SEEDS = list(PAPER_SEEDS)
FAIR_PROTOCOL_VERSION = "neurocomp-fair-v1"
COMMON_PROTOCOL_SHA256 = "445d12940d051634680727dc3739822c4030a90d7a3a3a90894f1d42e490d2fe"
METHOD_IMPLEMENTATION_VERSION = "fesem-standalone-v1"
CALD_REVISION_VERSION = METHOD_IMPLEMENTATION_VERSION

from collections import Counter, defaultdict


## 3. Select the experiment batch

Run any subset of the documented grid. Real-data experiments always use the complete 100-round budget. Set `SMOKE_TEST=True` for the short synthetic execution check.

In [ ]:
DATASETS = ["cifar10", "emnist"]
ALPHAS = [0.3, 0.5, 0.7]
SEEDS = [42, 101, 2024, 3407, 8888]
VARIANTS = ["full"]
METHOD = "fesem"
SMOKE_TEST = os.environ.get("LATENTFED_SMOKE", "0") == "1"
RESUME = True
INCLUDE_CHECKPOINTS_IN_ZIP = False
default_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_ROOT = Path(os.environ.get("LATENTFED_OUTPUT_ROOT", str(default_root / "09_fesem")))
print("Mode:", "synthetic smoke check" if SMOKE_TEST else "real-data paper grid")
print("Outputs:", OUTPUT_ROOT)


## 4. Experiment configuration

The common client and training budget is fixed; method-specific settings are explicit.

In [ ]:
@dataclass
class ExperimentConfig:
    dataset: str = "cifar10"
    seed: int = 42
    alpha: float = 0.3
    method: str = "fesem"
    num_clients: int = 20
    rounds: int = 100
    participation_ratio: float = 1.0
    local_epochs: int = 1
    batch_size: int = 32
    num_workers: int = 2
    classifier_lr_cifar10: float = 0.0005
    classifier_lr_emnist: float = 0.0001
    weight_decay: float = 0.0001
    num_centers: int = 3
    center_update_every: int = 150
    clients_per_cluster: int = -1
    mode: str = "no_size"
    regul_term: float = 0.1
    mu: float = 0.1
    variant: str = "full"
    data_root: str = "./data"
    validation_fraction: float = 0.1
    min_train_samples: int = 10
    min_validation_samples: int = 1
    min_test_samples: int = 1
    partition_max_attempts: int = 100
    evaluation_split: str = "test"
    summary_view: str = "deterministic"
    allow_custom_protocol: bool = False
    smoke_test: bool = False
    checkpoint_every: int = 5

    @property
    def classifier_lr(self):
        return (
            self.classifier_lr_cifar10 if self.dataset == "cifar10" else self.classifier_lr_emnist
        )

    @property
    def run_name(self) -> str:
        dist = "iid" if self.alpha is None else f"alpha_{self.alpha}"
        return f"{self.dataset}_{dist}_{self.method}_seed_{self.seed}"

    @property
    def config_id(self):
        excluded = {"data_root", "checkpoint_every", "num_workers"}
        payload = {k: v for k, v in asdict(self).items() if k not in excluded}
        payload.update(
            protocol_version=FAIR_PROTOCOL_VERSION,
            common_code=COMMON_PROTOCOL_SHA256,
            method_code=METHOD_IMPLEMENTATION_VERSION,
        )
        return hashlib.sha256(
            json.dumps(payload, sort_keys=True, allow_nan=False).encode()
        ).hexdigest()[:12]

    def validate(self):
        validate_common_config(self)
        if (self.num_clients, self.participation_ratio, self.local_epochs, self.batch_size) != (
            20,
            1.0,
            1,
            32,
        ):
            raise ValueError(
                "Use 20 clients, full participation, one local epoch and batch size 32."
            )
        if self.evaluation_split != "test" or self.validation_fraction != 0.1:
            raise ValueError("Use the shared validation holdout and official test split.")
        if self.method != "fesem" or self.variant != "full":
            raise ValueError("This notebook implements FeSEM.")
        if self.alpha not in PAPER_ALPHAS:
            raise ValueError("Choose a documented Dirichlet concentration.")
        if not self.allow_custom_protocol and (self.rounds != 100 or self.seed not in PAPER_SEEDS):
            raise ValueError("Use 100 rounds and an official experiment seed.")
        if (self.num_centers, self.center_update_every, self.clients_per_cluster) != (3, 150, -1):
            raise ValueError("Use the documented three-center FeSEM configuration.")


## 5. Seeded random streams

Independent roles keep client data and optimization streams reproducible.

In [ ]:
def set_seed(seed: int) -> None:
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def state_dict_to_cpu(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in state.items()}


def weighted_average_states(
    states: list[dict[str, torch.Tensor]], weights: list[float]
) -> dict[str, torch.Tensor]:
    total = float(np.sum(weights))
    if total <= 0:
        weights = [1.0 / len(states)] * len(states)
    else:
        weights = [float(w) / total for w in weights]
    out: dict[str, torch.Tensor] = {}
    for key in states[0].keys():
        if not torch.is_floating_point(states[0][key]):
            out[key] = states[0][key].detach().cpu().clone()
            continue
        acc = None
        for state, weight in zip(states, weights):
            value = state[key].detach().float().cpu() * weight
            acc = value if acc is None else acc + value
        out[key] = acc
    return out


def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + eps)


def role_seed(seed, *roles):
    value = json.dumps([int(seed), *roles], separators=(",", ":"))
    return int.from_bytes(hashlib.sha256(value.encode()).digest()[:4], "little")


@contextmanager
def cpu_random_stream(seed):
    """Isolate augmentation/model construction from other algorithm activity."""
    python_state, numpy_state = (random.getstate(), np.random.get_state())
    try:
        with torch.random.fork_rng(devices=[]):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)


def array_hash(arrays):
    digest = hashlib.sha256()
    for key, value in sorted(arrays.items()):
        a = np.ascontiguousarray(value)
        digest.update(key.encode())
        digest.update(str(a.dtype).encode())
        digest.update(json.dumps(a.shape).encode())
        digest.update(a.tobytes())
    return digest.hexdigest()


def common_selection(cfg, round_idx):
    count = max(1, int(round(cfg.num_clients * cfg.participation_ratio)))
    rng = np.random.default_rng(role_seed(cfg.seed, "participants", cfg.num_clients, round_idx))
    return sorted(rng.choice(cfg.num_clients, size=count, replace=False).tolist())


def validate_common_config(cfg):
    if not cfg.allow_custom_protocol:
        if (cfg.num_clients, cfg.participation_ratio) not in {(20, 1.0), (100, 0.1)}:
            raise ValueError(
                "Use set_scenario('20_full') or set_scenario('100_random10'); custom studies require allow_custom_protocol=True"
            )
        expected = {
            "local_epochs": 1,
            "batch_size": 32,
            "classifier_lr_cifar10": 0.0005,
            "classifier_lr_emnist": 0.0001,
            "weight_decay": 0.0001,
        }
        for key, value in expected.items():
            if getattr(cfg, key) != value:
                raise ValueError(
                    f"Common benchmark requires {key}={value}; declare a custom protocol for another setting"
                )
        if getattr(cfg, "local_optimizer", "adam").lower() != "adam":
            raise ValueError("The common classifier optimizer is Adam")
    if getattr(cfg, "summary_view", "deterministic") != "deterministic":
        raise ValueError("The common protocol requires a deterministic training summary view")
    if cfg.dataset not in {"cifar10", "emnist"} or cfg.evaluation_split not in {
        "validation",
        "test",
    }:
        raise ValueError("Unknown dataset/evaluation split")
    if not 0 < cfg.validation_fraction < 1 or not 0 < cfg.participation_ratio <= 1:
        raise ValueError("Invalid validation or participation fraction")
    if cfg.alpha is not None and (not np.isfinite(cfg.alpha) or cfg.alpha <= 0):
        raise ValueError("Dirichlet alpha must be positive or None for IID")
    if (
        min(
            cfg.num_clients,
            cfg.rounds,
            cfg.batch_size,
            cfg.local_epochs,
            cfg.partition_max_attempts,
            cfg.min_train_samples,
            cfg.min_validation_samples,
            cfg.min_test_samples,
            cfg.checkpoint_every,
        )
        < 1
    ):
        raise ValueError("Training budgets and partition minimums must be positive")
    if (
        cfg.participation_ratio < 1
        and getattr(cfg, "reassign_all_clients_each_round", False)
        and (cfg.method in {"ifca", "hypcluster"})
    ):
        raise ValueError(
            "Loss-based reassignment may contact only selected clients in the partial-participation protocol"
        )


## 6. Dataset loading

CIFAR-10 uses a pinned Hugging Face archive with checksums. EMNIST uses the 47-class Balanced split with orientation correction.

In [ ]:
CIFAR10_HF_REPO = "MIT-OL-AI-D/cifar-10-python"

CIFAR10_HF_REVISION = "a48007227f9e2cd0af96175f4afb5ac4965e261b"

CIFAR10_HF_URL = f"https://huggingface.co/datasets/{CIFAR10_HF_REPO}/resolve/{CIFAR10_HF_REVISION}/cifar-10-python.tar.gz"

CIFAR10_ARCHIVE_MD5 = "c58f30108f718f92721af3b95e74349a"


class EMNISTFixOrientation:

    def __call__(self, img):
        return TF.hflip(TF.rotate(img, -90))


def download_cifar10(data_root):
    """Download the original archive from a pinned HF mirror, retaining sample order."""
    from torchvision.datasets import CIFAR10
    from torchvision.datasets.utils import check_integrity, download_and_extract_archive

    root = Path(data_root).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    files = [
        *CIFAR10.train_list,
        *CIFAR10.test_list,
        (CIFAR10.meta["filename"], CIFAR10.meta["md5"]),
    ]

    def verified():
        return all((check_integrity(root / CIFAR10.base_folder / name, md5) for name, md5 in files))

    if not verified():
        print(f"Downloading/verifying CIFAR-10 from Hugging Face: {CIFAR10_HF_URL}")
        download_and_extract_archive(
            CIFAR10_HF_URL, str(root), filename="cifar-10-python.tar.gz", md5=CIFAR10_ARCHIVE_MD5
        )
        if not verified():
            raise RuntimeError("CIFAR-10 mirror files failed torchvision integrity verification")
    return str(root)


def get_dataset_bundle(dataset_name: str, data_root: str = "./data"):
    dataset_name = dataset_name.lower()
    if dataset_name == "cifar10":
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.2435, 0.2616)),
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.2435, 0.2616)),
            ]
        )
        cifar_root = download_cifar10(data_root)
        train_ds = datasets.CIFAR10(
            root=cifar_root, train=True, download=False, transform=train_transform
        )
        test_ds = datasets.CIFAR10(
            root=cifar_root, train=False, download=False, transform=test_transform
        )
        labels_train = np.asarray(train_ds.targets)
        labels_test = np.asarray(test_ds.targets)
        print(
            f"CIFAR-10 ready from verified Hugging Face cache at {cifar_root}: {len(train_ds):,} train / {len(test_ds):,} test"
        )
        return (train_ds, test_ds, labels_train, labels_test, 3, 32, 10)
    if dataset_name == "emnist":
        train_transform = transforms.Compose(
            [
                EMNISTFixOrientation(),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        test_transform = transforms.Compose(
            [
                EMNISTFixOrientation(),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        train_ds = datasets.EMNIST(
            root=data_root, split="balanced", train=True, download=True, transform=train_transform
        )
        test_ds = datasets.EMNIST(
            root=data_root, split="balanced", train=False, download=True, transform=test_transform
        )
        labels_train = np.asarray(train_ds.targets)
        labels_test = np.asarray(test_ds.targets)
        return (train_ds, test_ds, labels_train, labels_test, 1, 28, 47)
    raise ValueError(f"Unsupported dataset: {dataset_name}")


## 7. Matched client partitions

Allocate the official train/test samples across 20 clients and reserve the shared within-client validation holdout.

In [ ]:
def allocate_counts(count, probabilities):
    expected = count * probabilities
    counts = np.floor(expected).astype(np.int64)
    order = np.argsort(-(expected - counts), kind="stable")
    counts[order[: count - counts.sum()]] += 1
    return counts


def make_common_partitions(train_labels, test_labels, cfg):
    """One class-to-client probability matrix for both official data splits."""
    train_labels = np.asarray(train_labels, dtype=np.int64)
    test_labels = np.asarray(test_labels, dtype=np.int64)
    classes = np.unique(train_labels)
    if (
        not np.array_equal(classes, np.arange(len(classes)))
        or not np.isin(test_labels, classes).all()
    ):
        raise ValueError(
            "Labels must be contiguous zero-based classes, with test classes present in training"
        )
    if (
        len(train_labels) < cfg.num_clients * (cfg.min_train_samples + cfg.min_validation_samples)
        or len(test_labels) < cfg.num_clients * cfg.min_test_samples
    ):
        raise ValueError("Dataset is too small for the requested client minimums")
    n = cfg.num_clients
    for attempt in range(cfg.partition_max_attempts):
        rng = np.random.default_rng(role_seed(cfg.seed, "allocation", attempt))
        probabilities = (
            np.full((len(classes), n), 1 / n)
            if cfg.alpha is None
            else rng.dirichlet(np.full(n, cfg.alpha), size=len(classes))
        )
        groups = {split: [[] for _ in range(n)] for split in ("train", "validation", "test")}
        for cls in classes:
            train_idx = np.where(train_labels == cls)[0]
            test_idx = np.where(test_labels == cls)[0]
            rng.shuffle(train_idx)
            rng.shuffle(test_idx)
            train_chunks = np.split(
                train_idx, np.cumsum(allocate_counts(len(train_idx), probabilities[cls]))[:-1]
            )
            test_chunks = np.split(
                test_idx, np.cumsum(allocate_counts(len(test_idx), probabilities[cls]))[:-1]
            )
            for client, (tr, te) in enumerate(zip(train_chunks, test_chunks)):
                nv = (
                    min(max(1, int(round(len(tr) * cfg.validation_fraction))), len(tr) - 1)
                    if len(tr) >= 2
                    else 0
                )
                groups["validation"][client].extend(tr[:nv].tolist())
                groups["train"][client].extend(tr[nv:].tolist())
                groups["test"][client].extend(te.tolist())
        minimums = {
            "train": cfg.min_train_samples,
            "validation": cfg.min_validation_samples,
            "test": cfg.min_test_samples,
        }
        if all((min(map(len, groups[s])) >= minimums[s] for s in groups)):
            arrays = {
                f"{split}_{i}": np.asarray(rng.permutation(ids), dtype=np.int64)
                for split, clients in groups.items()
                for i, ids in enumerate(clients)
            }
            arrays["allocation"] = probabilities
            return (arrays, attempt + 1)
    raise ValueError("Partition minimums were not met; no invalid/empty-client split was returned")


def verify_partitions(arrays, train_labels, test_labels, cfg):
    allocation = arrays["allocation"]
    if (
        allocation.shape != (len(np.unique(train_labels)), cfg.num_clients)
        or not np.isfinite(allocation).all()
        or (allocation < 0).any()
        or (not np.allclose(allocation.sum(axis=1), 1))
    ):
        raise ValueError("Invalid shared allocation probabilities")
    for splits, labels in [(("train", "validation"), train_labels), (("test",), test_labels)]:
        indices = np.concatenate(
            [arrays[f"{s}_{i}"] for s in splits for i in range(cfg.num_clients)]
        )
        if not np.array_equal(np.sort(indices), np.arange(len(labels))):
            raise ValueError("Partitions have missing, duplicate or out-of-range indices")
    for split, minimum in [
        ("train", cfg.min_train_samples),
        ("validation", cfg.min_validation_samples),
        ("test", cfg.min_test_samples),
    ]:
        if min((len(arrays[f"{split}_{i}"]) for i in range(cfg.num_clients))) < minimum:
            raise ValueError("Cached client minimum is invalid")


## 8. Paired loaders and initialization

Training, deterministic assignment views and official test loaders share the LatentFed partition and initialization protocol.

In [ ]:
class PairedTrainingView(Dataset):

    def __init__(self, dataset, indices, seed, client, round_idx, epoch):
        self.dataset, self.indices = (dataset, list(map(int, indices)))
        self.seed, self.client, self.round_idx, self.epoch = (seed, client, round_idx, epoch)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        raw_index = self.indices[index]
        with cpu_random_stream(
            role_seed(self.seed, "augmentation", self.client, self.round_idx, self.epoch, raw_index)
        ):
            return self.dataset[raw_index]


class PairedClientLoader:
    """Private per-client/round/epoch shuffle and augmentation streams."""

    def __init__(self, dataset, indices, cfg, client):
        self.source, self.indices, self.cfg, self.client = (dataset, indices, cfg, client)
        self.dataset = Subset(dataset, indices.tolist())
        self.set_round(0)

    def set_round(self, round_idx):
        self.round_idx, self.epoch = (round_idx, 0)

    def __iter__(self):
        epoch = self.epoch
        self.epoch += 1
        view = PairedTrainingView(
            self.source, self.indices, self.cfg.seed, self.client, self.round_idx, epoch
        )
        generator = torch.Generator().manual_seed(
            role_seed(self.cfg.seed, "minibatches", self.client, self.round_idx, epoch)
        )
        return iter(
            DataLoader(
                view,
                batch_size=self.cfg.batch_size,
                shuffle=True,
                num_workers=self.cfg.num_workers,
                generator=generator,
                pin_memory=torch.cuda.is_available(),
                drop_last=False,
            )
        )

    def __len__(self):
        return (len(self.indices) + self.cfg.batch_size - 1) // self.cfg.batch_size


def deterministic_loaders(dataset, parts, cfg):
    return [
        DataLoader(
            Subset(dataset, ids.tolist()),
            batch_size=cfg.batch_size * 4,
            shuffle=False,
            num_workers=cfg.num_workers,
            generator=torch.Generator().manual_seed(role_seed(cfg.seed, "evaluation-loader", i)),
            pin_memory=torch.cuda.is_available(),
        )
        for i, ids in enumerate(parts)
    ]


def isolated_model_state(factory, cfg, role):
    with cpu_random_stream(role_seed(cfg.seed, role, cfg.dataset)):
        model = factory()
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


class FairRunData:
    """Preserved partition/initialization, with metadata held in memory."""

    def __init__(self, cfg, bundle):
        train, test, tr_y, te_y, channels, size, classes = bundle
        arrays, attempts = make_common_partitions(tr_y, te_y, cfg)
        verify_partitions(arrays, tr_y, te_y, cfg)
        self.arrays = arrays
        self.train_parts, self.validation_parts, self.test_parts = [
            [arrays[f"{s}_{i}"] for i in range(cfg.num_clients)]
            for s in ("train", "validation", "test")
        ]
        self.base_state = isolated_model_state(
            lambda: SimpleCNN(channels, classes), cfg, "classifier-initialization"
        )
        view = copy.copy(train)
        view.transform = test.transform
        self.train_loaders = [
            PairedClientLoader(train, ids, cfg, i) for i, ids in enumerate(self.train_parts)
        ]
        self.assignment_loaders = deterministic_loaders(view, self.train_parts, cfg)
        self.evaluation_loaders = deterministic_loaders(
            test if cfg.evaluation_split == "test" else view,
            self.test_parts if cfg.evaluation_split == "test" else self.validation_parts,
            cfg,
        )
        self.schedule = [common_selection(cfg, r) for r in range(1, cfg.rounds + 1)]
        self.manifest = dict(
            config=asdict(cfg),
            partition_hash=array_hash(arrays),
            partition_attempts=attempts,
            initial_state_hash=array_hash({k: v.numpy() for k, v in self.base_state.items()}),
            schedule_hash=array_hash({"participants": np.asarray(self.schedule)}),
            packages={
                p: importlib.metadata.version(p)
                for p in (
                    "torch",
                    "torchvision",
                    "numpy",
                    "pandas",
                    "scipy",
                    "scikit-learn",
                    "hdbscan",
                )
            },
            device=str(get_device()),
            cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            deterministic_algorithms=True,
            classifier_pooling="equivalent_linear_adaptive_average_v1",
        )

    def set_round(self, round_idx):
        for loader in self.train_loaders:
            loader.set_round(round_idx)


## 9. Classifier

The shared three-block CNN uses deterministic adaptive-average pooling.

In [ ]:
class DeterministicAdaptiveAvgPool2d(nn.AdaptiveAvgPool2d):
    """Same adaptive averaging bins and state_dict as nn.AdaptiveAvgPool2d.

    Avoid its unsupported deterministic CUDA backward: express the fixed linear
    average as F.linear. No trainable weights, RNG calls or model-state changes.
    CUBLAS_WORKSPACE_CONFIG is set in the dependency cell for strict CUDA matmul.
    """

    def forward(self, x):
        if x.ndim not in (3, 4):
            raise ValueError("AdaptiveAvgPool2d expects CHW or NCHW input")
        height, width = x.shape[-2:]
        target = (
            (self.output_size, self.output_size)
            if isinstance(self.output_size, int)
            else self.output_size
        )
        out_h = height if target[0] is None else target[0]
        out_w = width if target[1] is None else target[1]
        if min(height, width, out_h, out_w) < 1:
            raise ValueError("Pooling dimensions must be positive")
        if (height, width) == (out_h, out_w):
            return x

        def bin_weights(input_size, output_size):
            output = torch.arange(output_size, device=x.device)
            start = torch.div(output * input_size, output_size, rounding_mode="floor")
            end = torch.div(
                (output + 1) * input_size + output_size - 1, output_size, rounding_mode="floor"
            )
            positions = torch.arange(input_size, device=x.device)
            mask = (positions[None, :] >= start[:, None]) & (positions[None, :] < end[:, None])
            return mask.to(x.dtype) / (end - start).to(x.dtype)[:, None]

        rows, columns = (bin_weights(height, out_h), bin_weights(width, out_w))
        weights = (rows[:, None, :, None] * columns[None, :, None, :]).reshape(
            out_h * out_w, height * width
        )
        return F.linear(x.flatten(-2), weights).reshape(*x.shape[:-2], out_h, out_w)


class SimpleCNN(nn.Module):

    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            DeterministicAdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


## 10. FeSEM centers and assignments

Initialize three centers using the released first-K Lloyd procedure. Center updates use the supplied uniform aggregation rule.

In [ ]:
class UpstreamKMeans:
    """NumPy port of kmean_model.py: first K points, 5 initial steps, then 1.
    An empty assignment resets first-center initialization for the next call.
    """

    def __init__(self, num_centers):
        self.num_centers, self.first, self.centroids = (num_centers, True, None)

    def assign_clusters(self, points):
        x = np.asarray(points, dtype=np.float32)
        iterations = 1
        if self.first:
            self.centroids = x[: self.num_centers].copy()
            iterations, self.first = (5, False)
        for _ in range(iterations):
            distances = np.sum(np.square(x[:, None, :] - self.centroids[None, :, :]), axis=2)
            labels = np.argmin(distances, axis=1).astype(np.int64)
            numerator = np.zeros_like(self.centroids)
            denominator = np.zeros_like(self.centroids)
            for row, cluster in zip(x, labels):
                numerator[cluster] += row
                denominator[cluster] += 1
            with np.errstate(invalid="ignore", divide="ignore"):
                self.centroids = numerator / denominator
        if any((np.sum(labels == k) == 0 for k in range(self.num_centers))):
            self.first = True
        return labels


def initialize_fesem(cfg, in_channels, num_classes, initial_state):
    base = SimpleCNN(in_channels, num_classes)
    base_state = copy.deepcopy(initial_state)
    states = [copy.deepcopy(base_state) for _ in range(cfg.num_clients)]
    centers = {k: copy.deepcopy(base_state) for k in range(cfg.num_centers)}
    np.random.seed(cfg.seed + 20)
    client_order = np.random.choice(np.arange(cfg.num_clients), cfg.num_clients, replace=False)
    feature_dim = int(base_state["classifier.3.weight"].numel())
    points = np.random.uniform(-0.01, 0.01, (cfg.num_clients, feature_dim))
    clusterer = UpstreamKMeans(cfg.num_centers)
    labels = clusterer.assign_clusters(points)
    assignments = np.empty(cfg.num_clients, dtype=np.int64)
    assignments[client_order] = labels
    return (states, centers, assignments, client_order, points, clusterer)


def aggregate_selected_updates(updates):
    result = {}
    for key, value in updates[0]["classifier_state"].items():
        if torch.is_floating_point(value):
            total = sum((u["classifier_state"][key].double() * 2.0 for u in updates))
            result[key] = (total / (2.0 * len(updates))).to(value.dtype)
        else:
            result[key] = value.clone()
    return result


def recluster_upstream(points, client_order, updates, clusterer):
    positions = {int(client): row for row, client in enumerate(client_order)}
    for update in updates:
        points[positions[update["client_id"]]] = (
            update["classifier_state"]["classifier.3.weight"].T.reshape(-1).numpy()
        )
    labels = clusterer.assign_clusters(points)
    assignments = np.empty(len(client_order), dtype=np.int64)
    assignments[client_order] = labels
    return assignments


## 11. Local training

Run one local epoch with Adam and the shared dataset-specific learning rate.

In [ ]:
def train_one_client(client_id, classifier_state, loader, cfg, in_channels, num_classes, device):
    classifier = SimpleCNN(in_channels, num_classes).to(device)
    classifier.load_state_dict(classifier_state)
    classifier.train()
    optimizer = torch.optim.Adam(
        classifier.parameters(), lr=cfg.classifier_lr, weight_decay=cfg.weight_decay
    )
    total_seen, correct, loss_total, ce_total = (0, 0, 0.0, 0.0)
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            x, y = (x.to(device, non_blocking=True), y.to(device, non_blocking=True))
            optimizer.zero_grad(set_to_none=True)
            logits = classifier(x)
            ce = F.cross_entropy(logits, y)
            loss = ce
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite FeSEM objective on client {client_id}")
            loss.backward()
            optimizer.step()
            n = int(y.numel())
            total_seen += n
            correct += int((logits.argmax(1) == y).sum().item())
            loss_total += float(loss.detach()) * n
            ce_total += float(ce.detach()) * n
    if total_seen == 0:
        raise ValueError("Empty training client")
    return dict(
        client_id=client_id,
        classifier_state=state_dict_to_cpu(classifier.state_dict()),
        num_train=len(loader.dataset),
        num_seen=total_seen,
        train_loss=loss_total / total_seen,
        train_ce=ce_total / total_seen,
        train_acc=correct / total_seen,
    )


## 12. Official test evaluation

Record correct/total counts for each client and compute the method-specific predictive and cluster metrics.

In [ ]:
@torch.no_grad()
def evaluate_model(
    model: nn.Module, loader: DataLoader, device: torch.device
) -> tuple[int, int, float]:
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
        loss_sum += float(loss.item())
    return (correct, total, loss_sum / max(1, total))


def metrics_from_client_rows(rows):
    if not rows or any((row["total"] <= 0 for row in rows)):
        raise ValueError("Accuracy requires a nonempty test set for every client")
    groups = {}
    for row in rows:
        group = groups.setdefault(int(row["cluster_id"]), dict(correct=0, total=0, clients=0))
        group["correct"] += row["correct"]
        group["total"] += row["total"]
        group["clients"] += 1
    cluster_rows = [
        dict(cluster_id=k, **v, accuracy=v["correct"] / v["total"])
        for k, v in sorted(groups.items())
    ]
    accuracies = [row["accuracy"] for row in cluster_rows]
    metrics = {
        "W-Acc": sum((row["correct"] for row in rows)) / sum((row["total"] for row in rows)),
        "M-Acc": float(np.mean([row["correct"] / row["total"] for row in rows])),
        "MC": float(np.mean(accuracies)),
        "BC": float(max(accuracies)),
        "WC": float(min(accuracies)),
    }
    return (metrics, cluster_rows)


@torch.no_grad()
def evaluate_assigned_models(centers, assignments, test_loaders, in_channels, num_classes, device):
    rows = []
    for client_id, loader in enumerate(test_loaders):
        cluster_id = int(assignments[client_id])
        model = SimpleCNN(in_channels, num_classes).to(device)
        model.load_state_dict(centers[cluster_id])
        correct, total, loss = evaluate_model(model, loader, device)
        rows.append(
            dict(
                client_id=client_id,
                cluster_id=cluster_id,
                correct=int(correct),
                total=int(total),
                accuracy=correct / total if total else float("nan"),
                loss=float(loss),
            )
        )
        del model
    metrics, cluster_rows = metrics_from_client_rows(rows)
    return (metrics, rows, cluster_rows)


def geometry_metrics(matrix, labels, previous=None):
    occupied = np.unique(labels)
    metrics = dict(
        K=int(len(occupied)),
        Silhouette=float("nan"),
        CH=float("nan"),
        DB=float("nan"),
        SSE=0.0,
        TemporalARI=(
            float("nan") if previous is None else float(adjusted_rand_score(previous, labels))
        ),
    )
    for cluster_id in occupied:
        points = matrix[labels == cluster_id]
        metrics["SSE"] += float(np.square(points - points.mean(axis=0)).sum())
    nondegenerate = np.any(np.ptp(matrix, axis=0) > 0)
    if 1 < len(occupied) < len(labels) and nondegenerate:
        metrics["Silhouette"] = float(silhouette_score(matrix, labels, metric="euclidean"))
        metrics["CH"] = float(calinski_harabasz_score(matrix, labels))
        metrics["DB"] = float(davies_bouldin_score(matrix, labels))
    return metrics


## 13. Experiment grid and diagnostic data

Select the official real-data grid or the two-round synthetic check.

In [ ]:
def make_config(
    dataset="cifar10",
    alpha=0.3,
    seed=42,
    variant="full",
    method="fesem",
    data_root="./data",
    smoke=False,
):
    if (
        dataset not in PAPER_DATASETS
        or alpha not in PAPER_ALPHAS
        or variant != "full"
        or method != "fesem"
    ):
        raise ValueError("Choose a documented dataset, concentration and seed for FeSEM.")
    cfg = ExperimentConfig(
        dataset=dataset,
        alpha=alpha,
        seed=7 if smoke else seed,
        data_root=str(data_root),
        rounds=2 if smoke else 100,
        num_workers=0 if smoke else 2,
        allow_custom_protocol=smoke,
        min_train_samples=1 if smoke else 10,
    )
    cfg.smoke_test = smoke
    cfg.validate()
    return cfg


def experiment_plan(
    datasets=PAPER_DATASETS, alphas=PAPER_ALPHAS, seeds=PAPER_SEEDS, variants=("full",), smoke=False
):
    if not datasets or not alphas or (not seeds) or (not variants):
        raise ValueError("The grid may not be empty.")
    if any((x not in PAPER_DATASETS for x in datasets)) or any(
        (x not in PAPER_ALPHAS for x in alphas)
    ):
        raise ValueError("Grid contains an unsupported dataset or alpha.")
    if any((x not in PAPER_SEEDS for x in seeds)) or any(
        (x not in SUPPORTED_VARIANTS for x in variants)
    ):
        raise ValueError("Grid contains an unsupported official seed or variant.")
    selected_alphas = (alphas[0],) if smoke else alphas
    selected_seeds = (7,) if smoke else seeds
    return [
        dict(dataset=d, alpha=a, seed=s, variant=v)
        for d in datasets
        for a in selected_alphas
        for s in selected_seeds
        for v in variants
    ]


class SyntheticImages(Dataset):
    """Small deterministic tensors used only by the explicitly labelled smoke mode."""

    def __init__(self, count, channels, size, classes, seed):
        generator = torch.Generator().manual_seed(seed)
        self.images = torch.randn(count, channels, size, size, generator=generator)
        self.targets = np.arange(count) % classes
        self.transform = None

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return (self.images[index], int(self.targets[index]))


def synthetic_bundle(dataset):
    channels, size, classes = (3, 32, 10) if dataset == "cifar10" else (1, 28, 47)
    train = SyntheticImages(400, channels, size, classes, 1739)
    test = SyntheticImages(200, channels, size, classes, 2718)
    return (train, test, train.targets, test.targets, channels, size, classes)


## 14. Checkpoint serialization and code identity

Save local state, inspectable metrics and portable result archives.

In [ ]:
def clean(value):
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean(x) for x in value]
    if isinstance(value, np.ndarray):
        return clean(value.tolist())
    if isinstance(value, np.generic):
        return clean(value.item())
    if isinstance(value, float) and (not math.isfinite(value)):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def json_write(path, value):
    Path(path).write_text(json.dumps(clean(value), indent=2, sort_keys=True, allow_nan=False))


def atomic_checkpoint(path, state):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    os.replace(temporary, path)


def load_checkpoint(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def _code_signature(code):
    def constant(value):
        if isinstance(value, types.CodeType):
            return _code_signature(value)
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if isinstance(value, (tuple, list)):
            return [constant(v) for v in value]
        if isinstance(value, frozenset):
            return sorted([constant(v) for v in value], key=repr)
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)

    return dict(
        bytecode=code.co_code.hex(),
        constants=[constant(v) for v in code.co_consts],
        names=code.co_names,
        variables=code.co_varnames,
        freevars=code.co_freevars,
        cellvars=code.co_cellvars,
        arguments=code.co_argcount,
        keyword_only=code.co_kwonlyargcount,
        positional_only=code.co_posonlyargcount,
        exception_table=code.co_exceptiontable.hex(),
        flags=code.co_flags,
    )


def code_digest():
    """Hash live notebook definitions independently of cell filenames/numbers."""
    definitions = {}
    for name, value in sorted(list(globals().items())):
        if inspect.isfunction(value) and value.__module__ == __name__:
            function = inspect.unwrap(value)
            definitions[name] = dict(
                code=_code_signature(function.__code__),
                defaults=clean(function.__defaults__),
                keyword_defaults=clean(function.__kwdefaults__),
            )
        elif inspect.isclass(value) and value.__module__ == __name__:
            methods = {}
            for key, member in value.__dict__.items():
                if isinstance(member, property):
                    member = member.fget
                elif isinstance(member, (classmethod, staticmethod)):
                    member = member.__func__
                if inspect.isfunction(member):
                    methods[key] = _code_signature(inspect.unwrap(member).__code__)
            definitions[name] = methods
    return hashlib.sha256(
        json.dumps(definitions, sort_keys=True, allow_nan=False).encode()
    ).hexdigest()


## 15. Metrics, five-seed aggregation and ZIP export

Save local state, inspectable metrics and portable result archives.

In [ ]:
def write_result(
    folder, cfg, history, manifest, method, smoke=False, complete=False, summary=None, extra=None
):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    from dataclasses import asdict

    json_write(folder / "config.json", asdict(cfg))
    clients = []
    diagnostics = []
    flat = []
    for row in history:
        flat.append(
            {k: v for k, v in row.items() if not isinstance(v, (dict, list, tuple, np.ndarray))}
        )
        for client in row.get("client_metrics", []):
            clients.append(dict(round=row["round"], **client))
        diagnostics.append(
            {k: v for k, v in row.items() if k not in flat[-1] and k != "client_metrics"}
        )
    pd.DataFrame(flat).to_csv(folder / "round_metrics.csv", index=False)
    pd.DataFrame(clients).to_csv(folder / "client_metrics.csv", index=False)
    with (folder / "round_diagnostics.jsonl").open("w") as f:
        for row, item in zip(history, diagnostics):
            f.write(json.dumps(clean(dict(round=row["round"], **item)), allow_nan=False) + "\n")
    json_write(
        folder / "manifest.json",
        dict(
            method=method,
            variant=cfg.variant,
            dataset=cfg.dataset,
            alpha=cfg.alpha,
            seed=cfg.seed,
            mode="smoke" if smoke else "paper",
            complete=complete,
            completed_rounds=len(history),
            publication_eligible=bool(complete and (not smoke) and (len(history) == 100)),
            code_sha256=code_digest(),
            protocol="matched_latentfed_v1",
            **manifest,
            **extra or {}
        ),
    )
    if summary is not None:
        json_write(folder / "seed_summary.json", summary)
    return folder


def export_zip(output, include_checkpoints=False):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    archive = output / "results.zip"
    temporary = output / "results.zip.tmp"
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(output.rglob("*")):
            if not p.is_file() or p in {archive, temporary}:
                continue
            if p.suffix not in {".json", ".csv", ".jsonl"} and (
                not (include_checkpoints and p.suffix == ".pt")
            ):
                continue
            z.write(p, str(p.relative_to(output)))
    temporary.replace(archive)
    return archive


def aggregate_runs(output):
    """Five-seed means/sample SD; never pools rounds or admits smoke results."""
    output = Path(output)
    groups = {}
    signatures = {}
    for path in output.rglob("seed_summary.json"):
        manifest = json.loads((path.parent / "manifest.json").read_text())
        if not manifest["publication_eligible"]:
            continue
        score = json.loads(path.read_text())
        if score["window_start"] != 91 or score["window_end"] != 100:
            raise ValueError("Wrong final window.")
        key = (manifest["method"], manifest["variant"], manifest["dataset"], manifest["alpha"])
        cfg = json.loads((path.parent / "config.json").read_text())
        scientific = {
            k: v
            for k, v in cfg.items()
            if k not in {"seed", "data_root", "num_workers", "checkpoint_every"}
        }
        signature = json.dumps(
            dict(
                configuration=scientific,
                code=manifest["code_sha256"],
                protocol=manifest["protocol"],
            ),
            sort_keys=True,
        )
        if key in signatures and signatures[key] != signature:
            raise ValueError("Seed aggregation would mix configurations or source versions.")
        signatures[key] = signature
        group = groups.setdefault(key, {})
        if manifest["seed"] in group:
            raise ValueError("Duplicate/conflicting seed in aggregate.")
        group[manifest["seed"]] = score
    summary = []
    coverage = []
    required = {42, 101, 2024, 3407, 8888}
    metrics = [
        "W-Acc",
        "M-Acc",
        "MC",
        "BC",
        "WC",
        "Silhouette",
        "Calinski-Harabasz",
        "Davies-Bouldin",
        "SSE",
        "Temporal ARI",
    ]
    for key, group in sorted(groups.items()):
        base = dict(zip(["method", "variant", "dataset", "alpha"], key))
        coverage.append(
            dict(
                **base,
                n_seeds=len(group),
                seeds=";".join(map(str, sorted(group))),
                complete_five_seed_group=set(group) == required
            )
        )
        if set(group) != required:
            continue
        row = dict(**base, n_seeds=5, std_ddof=1, window_start=91, window_end=100)
        for metric in metrics:
            vals = [group[s].get(metric) for s in sorted(required)]
            valid = all(
                (
                    v is not None
                    and math.isfinite(v)
                    and (group[s].get(metric + "_valid_rounds", 10) == 10)
                    for v, s in zip(vals, sorted(required))
                )
            )
            scale = 100 if metric in metrics[:5] else 1
            row[metric + "_mean"] = float(np.mean(vals)) * scale if valid else None
            row[metric + "_std"] = float(np.std(vals, ddof=1)) * scale if valid else None
        summary.append(row)
    pd.DataFrame(summary).to_csv(output / "five_seed_summary.csv", index=False)
    pd.DataFrame(coverage).to_csv(output / "seed_coverage.csv", index=False)
    return (pd.DataFrame(summary), pd.DataFrame(coverage))


## 16. Run identity and final-window scoring

Save local state, inspectable metrics and portable result archives.

In [ ]:
def _run_folder(output, method, cfg, smoke):
    folder = (
        Path(output)
        / ("smoke" if smoke else "paper")
        / method
        / cfg.variant
        / cfg.dataset
        / f"alpha_{cfg.alpha:g}"
        / f"seed_{cfg.seed}_{cfg.config_id}"
    )
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def _state_or_fresh(path, cfg, method, resume):
    if path.exists():
        if not resume:
            raise FileExistsError("Use RESUME=True or choose a fresh output directory.")
        saved = load_checkpoint(path)
        if saved.get("public_code_sha256") != code_digest():
            raise ValueError(
                "Checkpoint definitions differ. Use matching code or a fresh output directory."
            )
        if (
            saved.get("public_method") != method
            or ExperimentConfig(**saved["public_config"]).config_id != cfg.config_id
        ):
            raise ValueError("Checkpoint method or scientific settings differ.")
        return saved
    return dict(public_code_sha256=code_digest(), public_config=asdict(cfg), public_method=method)


def _score(history, cfg, method):
    if len(history) != 100 or [r["round"] for r in history] != list(range(1, 101)):
        raise ValueError("An endpoint requires all 100 ordered rounds.")
    score = dict(
        method=method,
        variant=cfg.variant,
        dataset=cfg.dataset,
        alpha=cfg.alpha,
        seed=cfg.seed,
        window_start=91,
        window_end=100,
    )
    metrics = [
        "W-Acc",
        "M-Acc",
        "MC",
        "BC",
        "WC",
        "Silhouette",
        "Calinski-Harabasz",
        "Davies-Bouldin",
        "SSE",
        "Temporal ARI",
    ]
    for metric in metrics:
        finite = [
            float(r[metric])
            for r in history[-10:]
            if r.get(metric) is not None and math.isfinite(float(r[metric]))
        ]
        score[metric] = sum(finite) / 10 if len(finite) == 10 else None
        score[metric + "_valid_rounds"] = len(finite)
        if metric in {"W-Acc", "M-Acc"} and len(finite) != 10:
            raise ValueError("Test accuracy is missing in the final window.")
    return score


def capture_rng_state():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


## 17. Client update and checkpoint orchestration

All 20 local updates and the complete sample budget are checked before a round is recorded. Resume restores both model state and random streams.

In [ ]:
def train_client(
    client, parent_state, fair, cfg, channels, classes, device, round_idx, prototypes=None
):
    set_seed(role_seed(cfg.seed, "local-stochastic", round_idx, client))
    arguments = dict(
        client_id=client,
        loader=fair.train_loaders[client],
        cfg=cfg,
        in_channels=channels,
        num_classes=classes,
        device=device,
    )
    parameters = inspect.signature(train_one_client).parameters
    state_name = next(
        name for name in ("classifier_state", "initial_state", "parent_state") if name in parameters
    )
    arguments[state_name] = parent_state
    if METHOD == "fedproto":
        arguments.update(global_prototypes=prototypes or {}, round_idx=round_idx)
    result = train_one_client(**arguments)
    expected = len(fair.train_parts[client]) * cfg.local_epochs
    seen = result.get("examples_seen", result.get("num_seen"))
    if seen is None:
        seen = round(result["num_train"] * cfg.local_epochs)
    if seen != expected:
        raise AssertionError("The client update must process its complete training partition.")
    result["examples_seen"] = seen
    return result


def restore_method_state(saved, fair):
    state = saved["state"]
    for key in ("partition_hash", "initial_state_hash", "schedule_hash", "packages", "device"):
        if state["manifest"][key] != fair.manifest[key]:
            raise ValueError("Resume data/runtime mismatch: " + key)
    restore_rng_state(state["rng"])
    return state


def normalize_metric_names(metrics):
    for original, standard in [
        ("CH", "Calinski-Harabasz"),
        ("DB", "Davies-Bouldin"),
        ("TemporalARI", "Temporal ARI"),
    ]:
        if original in metrics:
            metrics[standard] = metrics[original]
    return metrics


def run_baseline(cfg, output, smoke=False, resume=True, stop_after=None):
    cfg.validate()
    folder = _run_folder(output, METHOD, cfg, smoke)
    checkpoint = folder / "checkpoint.pt"
    saved = _state_or_fresh(checkpoint, cfg, METHOD, resume)
    set_seed(cfg.seed)
    bundle = (
        synthetic_bundle(cfg.dataset) if smoke else get_dataset_bundle(cfg.dataset, cfg.data_root)
    )
    _, _, _, labels_test, channels, _, classes = bundle
    fair = FairRunData(cfg, bundle)
    device = get_device()
    true_groups = np.asarray(
        [
            np.bincount(np.asarray(labels_test)[ids], minlength=classes).argmax()
            for ids in fair.test_parts
        ]
    )
    if "state" in saved:
        state = restore_method_state(saved, fair)
    else:
        state = initialize_method(fair, cfg, channels, classes, device)
        state.update(round=0, history=[], manifest=fair.manifest)
    final_round = cfg.rounds if stop_after is None else min(cfg.rounds, stop_after)
    try:
        for round_idx in range(state["round"] + 1, final_round + 1):
            fair.set_round(round_idx)
            updates, selected, diagnostics = method_round(
                state, fair, cfg, channels, classes, device, round_idx
            )
            if sorted(selected) != fair.schedule[round_idx - 1] or len(set(selected)) != 20:
                raise AssertionError("Every round must train all 20 clients exactly once.")
            examples = sum(u["examples_seen"] for u in updates)
            if examples != sum(map(len, fair.train_parts)) * cfg.local_epochs:
                raise AssertionError("The complete local training budget is required.")
            metrics, clients = evaluate_method(
                state, fair, cfg, channels, classes, device, true_groups
            )
            metrics = normalize_metric_names(metrics)
            losses = [u["train_loss"] for u in updates]
            weights = [u["examples_seen"] for u in updates]
            loss = (
                float(np.average(losses, weights=weights))
                if METHOD in {"fedproto", "fesem", "cfl"}
                else float(np.mean(losses))
            )
            state["history"].append(
                dict(
                    round=round_idx,
                    selected_clients=fair.schedule[round_idx - 1],
                    client_execution_order=selected,
                    trained_clients=20,
                    train_examples=examples,
                    evaluation_split="test",
                    train_loss=loss,
                    client_metrics=clients,
                    **metrics,
                    **diagnostics,
                )
            )
            state["round"] = round_idx
            if round_idx % cfg.checkpoint_every == 0 or round_idx == final_round:
                state["rng"] = capture_rng_state()
                atomic_checkpoint(checkpoint, {**saved, "state": state})
                write_result(
                    folder,
                    cfg,
                    state["history"],
                    state["manifest"],
                    METHOD,
                    smoke,
                    False,
                    extra=dict(
                        execution="serial single device", initialization=state["initialization"]
                    ),
                )
            if not smoke and (round_idx == 1 or round_idx % 10 == 0):
                print(
                    f"{METHOD} {cfg.dataset} round {round_idx}: W-Acc={metrics['W-Acc']:.4f}",
                    flush=True,
                )
    finally:
        state["rng"] = capture_rng_state()
        atomic_checkpoint(checkpoint, {**saved, "state": state})
    complete = state["round"] == cfg.rounds
    result = write_result(
        folder,
        cfg,
        state["history"],
        state["manifest"],
        METHOD,
        smoke,
        complete,
        _score(state["history"], cfg, METHOD) if complete and not smoke else None,
        extra=dict(
            execution="serial single device",
            initialization=state["initialization"],
            prediction_rule=METHOD_PREDICTION_RULE,
        ),
    )
    if METHOD == "cfl":
        json_write(folder / "threshold_calibration.json", state["calibration"].report())
    return result


## 18. FeSEM initialization and round update

The method-specific sequence is explicit: initialize state, perform local updates, update the server and evaluate the resulting models.

In [ ]:
METHOD_PREDICTION_RULE = "assigned_cluster_model"


def initialize_method(fair, cfg, channels, classes, device):
    bank, centers, assignments, order, points, clusterer = initialize_fesem(
        cfg, channels, classes, fair.base_state
    )
    return dict(
        bank=bank,
        centers=centers,
        assignments=assignments,
        order=order,
        points=points,
        clusterer=clusterer,
        previous_assignments=None,
        initialization=dict(kind="uniform_proxy_first_K_Lloyd", train_examples=0),
    )


def method_round(state, fair, cfg, channels, classes, device, round_idx):
    updates = []
    selected = []
    before = state["assignments"].copy()
    for cluster in range(cfg.num_centers):
        members = [
            int(client) for client in state["order"] if state["assignments"][client] == cluster
        ]
        if len(members) <= 1:
            continue
        active = (
            members
            if cfg.clients_per_cluster == -1
            else np.random.choice(
                members, min(cfg.clients_per_cluster, len(members)), replace=False
            ).tolist()
        )
        group = []
        for client in active:
            update = train_client(
                client, state["centers"][cluster], fair, cfg, channels, classes, device, round_idx
            )
            state["bank"][client] = update["classifier_state"]
            group.append(update)
            updates.append(update)
            selected.append(client)
        state["centers"][cluster] = aggregate_selected_updates(group)
    if round_idx != cfg.rounds and round_idx % cfg.center_update_every == 0:
        state["assignments"] = recluster_upstream(
            state["points"], state["order"], updates, state["clusterer"]
        )
    points = np.empty_like(state["points"], dtype=np.float32)
    points[state["order"]] = state["points"]
    diagnostics = geometry_metrics(points, state["assignments"], state["previous_assignments"])
    state["previous_assignments"] = state["assignments"].copy()
    return (
        updates,
        selected,
        dict(cluster_assignments=state["assignments"], training_assignments=before, **diagnostics),
    )


def evaluate_method(state, fair, cfg, channels, classes, device, true_groups):
    metrics, clients, _ = evaluate_assigned_models(
        state["centers"], state["assignments"], fair.evaluation_loaders, channels, classes, device
    )
    return metrics, clients


## 19. Experiment driver

Run the selected grid, checkpoint completed rounds and refresh the ZIP after each experiment.

In [ ]:
def run_grid(
    datasets=PAPER_DATASETS,
    alphas=PAPER_ALPHAS,
    seeds=PAPER_SEEDS,
    variants=("full",),
    output="outputs",
    smoke=False,
    resume=True,
    include_checkpoints=False,
    allow_cpu=False,
):
    if not smoke and not torch.cuda.is_available() and not allow_cpu:
        raise RuntimeError(
            "Enable a Kaggle GPU for paper mode, or set SMOKE_TEST=True for the short CPU check."
        )
    if smoke:
        torch.set_num_threads(1)
    plan = experiment_plan(datasets, alphas, seeds, variants, smoke=smoke)
    output = Path(output)
    mode_root = output / ("smoke" if smoke else "paper") / "fesem"
    mode_root.mkdir(parents=True, exist_ok=True)
    json_write(
        mode_root / "experiment_plan.json",
        dict(
            method="fesem",
            mode="smoke" if smoke else "paper",
            experiments=plan,
            clients=20,
            clients_per_round=20,
            rounds=2 if smoke else 100,
            local_epochs=1,
            batch_size=32,
        ),
    )
    folders = []
    try:
        for item in plan:
            cfg = make_config(**item, method="fesem", data_root=output / "data", smoke=smoke)
            print(
                f"Running {cfg.dataset} / {cfg.variant} / alpha={cfg.alpha} / seed={cfg.seed} / rounds={cfg.rounds}",
                flush=True,
            )
            folders.append(run_baseline(cfg, output, smoke=smoke, resume=resume))
            aggregate_runs(mode_root)
            export_zip(mode_root, include_checkpoints)
    finally:
        archive = export_zip(mode_root, include_checkpoints)
    return dict(runs=folders, archive=archive, summary=aggregate_runs(mode_root)[0])


## 20. Plotting helper

Display recorded accuracy and local-objective curves for the completed runs.

In [ ]:
def plot_run_preview(paths):
    """One last completed run per dataset/variant, clearly scoped as a preview."""
    chosen = {}
    for path in map(Path, paths):
        metadata = json.loads((path / "manifest.json").read_text())
        chosen[metadata["dataset"], metadata["variant"]] = (path, metadata)
    colors = {"cifar10": "#245B8A", "emnist": "#B76B18"}
    styles = {
        "full": ("-", "o"),
        "no_mdfe": ("--", "s"),
        "no_cald": (":", "^"),
        "ae_only": ("-.", "D"),
    }
    figure, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for (dataset, variant), (path, metadata) in chosen.items():
        frame = pd.read_csv(path / "round_metrics.csv")
        linestyle, marker = styles[variant]
        appearance = dict(
            color=colors[dataset],
            linestyle=linestyle,
            marker=marker,
            markersize=3,
            markevery=max(1, len(frame) // 8),
            linewidth=1.5,
            label=f"{dataset}: {variant}",
        )
        axes[0].plot(frame["round"], 100 * frame["W-Acc"], **appearance)
        axes[1].plot(frame["round"], frame["train_loss"], **appearance)
    axes[0].set(xlabel="Communication round", ylabel="Weighted test accuracy (%)", ylim=(0, 100))
    axes[1].set(xlabel="Communication round", ylabel="Recorded local training objective")
    for axis in axes:
        axis.grid(alpha=0.2)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), fontsize=8)
    smoke = all((m["mode"] == "smoke" for _, m in chosen.values()))
    figure.suptitle(
        "Synthetic smoke checks — not paper results"
        if smoke
        else "Last completed seed/configuration per dataset and variant"
    )
    figure.tight_layout(rect=(0, 0.14, 1, 0.98))
    return figure


## 21. Inspect the plan and models

Review the selected experiments and model dimensions.

In [ ]:
PLAN = experiment_plan(DATASETS, ALPHAS, SEEDS, VARIANTS, smoke=SMOKE_TEST)
display(pd.DataFrame(PLAN).head(12))
print("Experiments in this batch:", len(PLAN))
models = []
for dataset in DATASETS:
    cfg = make_config(dataset, ALPHAS[0], SEEDS[0], method=METHOD, smoke=SMOKE_TEST)
    channels, size, classes = (3, 32, 10) if dataset == "cifar10" else (1, 28, 47)
    model = SimpleCNN(channels, classes)
    models.append(
        dict(
            dataset=dataset,
            model=type(model).__name__,
            parameters=sum(p.numel() for p in model.parameters()),
            clients=cfg.num_clients,
            clients_per_round=20,
            rounds=cfg.rounds,
        )
    )
display(pd.DataFrame(models))


## 22. Run the selected experiments

Real-data runs use a CUDA GPU and the complete training budget.

In [ ]:
RESULT = run_grid(
    datasets=DATASETS,
    alphas=ALPHAS,
    seeds=SEEDS,
    variants=VARIANTS,
    output=OUTPUT_ROOT,
    smoke=SMOKE_TEST,
    resume=RESUME,
    include_checkpoints=INCLUDE_CHECKPOINTS_IN_ZIP,
)
print("Completed run folders:", len(RESULT["runs"]))


## 23. Results preview

Inspect saved metrics and recorded curves. Accuracy values in round logs are fractions.

In [ ]:
status = []
for folder in RESULT["runs"]:
    manifest = json.loads((folder / "manifest.json").read_text())
    status.append(
        {
            key: manifest[key]
            for key in [
                "method",
                "variant",
                "dataset",
                "alpha",
                "seed",
                "mode",
                "completed_rounds",
                "publication_eligible",
            ]
        }
    )
display(pd.DataFrame(status))
if RESULT["summary"].empty:
    print(
        "Seed endpoints are saved. Five-seed summaries are written when all five official seeds are complete."
    )
else:
    display(RESULT["summary"])
figure = plot_run_preview(RESULT["runs"])
display(figure)
plt.close(figure)


## 24. Download and resume

Download the ZIP from Kaggle Output. Include checkpoints when continuing in another session and restore the same output hierarchy to resume.

In [ ]:
mode_root = OUTPUT_ROOT / ("smoke" if SMOKE_TEST else "paper") / METHOD
DOWNLOAD_ZIP = export_zip(mode_root, include_checkpoints=INCLUDE_CHECKPOINTS_IN_ZIP)
link_path = (
    str(DOWNLOAD_ZIP)
    if Path("/kaggle/working").is_dir()
    else os.path.relpath(DOWNLOAD_ZIP, Path.cwd())
)
display(FileLink(link_path))
print("Download ZIP:", DOWNLOAD_ZIP)


## 25. FeSEM source license

The incorporated FeSEM routines retain their BSD 2-Clause license.

```text
BSD 2-Clause License

Copyright (c) 2018, TalwalkarLab
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
```